In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

params = pd.read_csv('../team_parameters.csv')
heatmap = pd.read_csv('../../data/heatmap_data_all_stadiums.csv')

cle_meta = params[params['team_code'] == 'CLE'].iloc[0]
cle = pd.read_csv(f"../../data/{cle_meta['dataset_file']}")
cle = cle[(cle['season'] >= int(cle_meta['data_start_year'])) & (cle['season'] <= int(cle_meta['data_end_year']))].copy()
cle = cle.dropna(subset=['temp_f', 'rhum', 'pres', 'wspd_mph', 'away_runs_scored', 'total_runs', 'strikeouts'])

cle['temp_bin'] = pd.qcut(cle['temp_f'], q=5, duplicates='drop')
cle['wind_bin'] = pd.qcut(cle['wspd_mph'], q=4, duplicates='drop')
cle['humidity_bin'] = pd.qcut(cle['rhum'], q=4, duplicates='drop')
cle['pressure_bin'] = pd.qcut(cle['pres'], q=5, duplicates='drop')

print(f"Cleveland source dataset: {cle_meta['dataset_file']} ({int(cle_meta['data_start_year'])}-{int(cle_meta['data_end_year'])})")
print(f'Cleveland home games with complete weather data: {len(cle)}')
cle[['game_date', 'away_team', 'temp_f', 'rhum', 'pres', 'wspd_mph', 'away_runs_scored', 'total_runs', 'strikeouts']].head()

In [ ]:
# Temperature Quintiles: Summary Table
temp_summary = cle.groupby('temp_bin', observed=True).agg(
    games=('temp_f', 'size'),
    away_runs_mean=('away_runs_scored', 'mean'),
    total_runs_mean=('total_runs', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
    hits_mean=('hits', 'mean'),
    home_runs_mean=('home_runs_hit', 'mean'),
    walk_mean=('walks', 'mean'),
    humidity_mean=('rhum', 'mean'),
    pressure_mean=('pres', 'mean'),
    wind_mean=('wspd_mph', 'mean')
).reset_index()

temp_summary['temp_range'] = temp_summary['temp_bin'].astype(str)
temp_summary[['games', 'away_runs_mean', 'total_runs_mean', 'strikeouts_mean', 'hits_mean', 'home_runs_mean', 'walk_mean', 'humidity_mean', 'pressure_mean', 'wind_mean']] = temp_summary[[
    'games', 'away_runs_mean', 'total_runs_mean', 'strikeouts_mean', 'hits_mean', 'home_runs_mean', 'walk_mean', 'humidity_mean', 'pressure_mean', 'wind_mean'
]].round(3)

temp_summary[['temp_range', 'games', 'away_runs_mean', 'total_runs_mean', 'strikeouts_mean', 'hits_mean', 'home_runs_mean', 'walk_mean', 'humidity_mean', 'pressure_mean', 'wind_mean']]

In [ ]:
# Temperature Quintiles: Runs, Strikeouts, and Weather Drift
temp_order = cle['temp_bin'].cat.categories
temp_labels = [str(b).replace(', ', '\n').replace('(', '').replace(']', '') for b in temp_order]
x = np.arange(len(temp_order))

plot_summary = temp_summary.copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0, 0].bar(x - 0.18, plot_summary['away_runs_mean'], width=0.36, color='#1f77b4', label='Away runs')
axes[0, 0].bar(x + 0.18, plot_summary['total_runs_mean'], width=0.36, color='#ff7f0e', label='Total runs')
axes[0, 0].set_title('Scoring by Temperature Quintile')
axes[0, 0].set_ylabel('Runs')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(temp_labels)
axes[0, 0].legend()

axes[0, 1].plot(x, plot_summary['strikeouts_mean'], marker='o', linewidth=2.5, color='#2ca02c', label='Strikeouts')
axes[0, 1].plot(x, plot_summary['hits_mean'], marker='s', linewidth=2.0, color='#d62728', label='Hits')
axes[0, 1].plot(x, plot_summary['home_runs_mean'], marker='^', linewidth=2.0, color='#9467bd', label='Home runs')
axes[0, 1].set_title('Contact Outcomes by Temperature Quintile')
axes[0, 1].set_ylabel('Per-game average')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(temp_labels)
axes[0, 1].legend()

axes[1, 0].plot(x, plot_summary['humidity_mean'], marker='o', linewidth=2.5, color='#17becf', label='Humidity')
axes[1, 0].plot(x, plot_summary['wind_mean'], marker='s', linewidth=2.5, color='#8c564b', label='Wind speed')
axes[1, 0].set_title('Do Other Weather Variables Shift With Temperature?')
axes[1, 0].set_ylabel('Humidity / Wind mph')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(temp_labels)
axes[1, 0].legend()

axes[1, 1].plot(x, plot_summary['pressure_mean'], marker='o', linewidth=2.5, color='#7f7f7f')
axes[1, 1].set_title('Average Pressure by Temperature Quintile')
axes[1, 1].set_ylabel('Pressure (hPa)')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(temp_labels)

for ax in axes.flat:
    ax.grid(alpha=0.25)

plt.suptitle('Progressive Field: Temperature-Led Weather Profile', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-Stadium Comparison: How Large Is Cleveland's Temperature Effect?
away_runs_temp = heatmap[(heatmap['metric'] == 'away_runs') & (heatmap['weather_variable'] == 'temp')].copy()
temp_effect_rank = away_runs_temp.groupby('team_code').agg(
    temp_effect_range=('diff_from_mean', lambda s: s.max() - s.min()),
    best_temp_bucket=('diff_from_mean', 'max'),
    worst_temp_bucket=('diff_from_mean', 'min')
).sort_values('temp_effect_range', ascending=False).reset_index()

temp_effect_rank['rank'] = np.arange(1, len(temp_effect_rank) + 1)
cle_rank = temp_effect_rank[temp_effect_rank['team_code'] == 'CLE']
display(cle_rank)
display(temp_effect_rank.head(10))

cle_temp_detail = away_runs_temp[away_runs_temp['team_code'] == 'CLE'].sort_values('diff_from_mean', ascending=False)
display(cle_temp_detail[['quintile_bin', 'diff_from_mean', 'n_games']])

In [ ]:
# Crossover Effects: Temperature x Wind / Humidity / Pressure
def crossover_summary(frame, other_bin, other_label):
    grouped = frame.groupby(['temp_bin', other_bin], observed=True).agg(
        games=('game_pk', 'size'),
        total_runs_mean=('total_runs', 'mean'),
        away_runs_mean=('away_runs_scored', 'mean'),
        strikeouts_mean=('strikeouts', 'mean')
    ).reset_index()
    grouped['temp_label'] = grouped['temp_bin'].astype(str)
    grouped[other_label] = grouped[other_bin].astype(str)
    return grouped

temp_wind = crossover_summary(cle, 'wind_bin', 'wind_label')
temp_humidity = crossover_summary(cle, 'humidity_bin', 'humidity_label')
temp_pressure = crossover_summary(cle, 'pressure_bin', 'pressure_label')

display(temp_wind)
display(temp_humidity)
display(temp_pressure)

In [ ]:
# Crossover Heatmaps
def draw_heatmap(ax, frame, row_col, col_col, value_col, title, cmap='YlOrRd'):
    pivot = frame.pivot(index=row_col, columns=col_col, values=value_col)
    values = pivot.values
    ax.imshow(values, aspect='auto', cmap=cmap)
    ax.set_title(title)
    ax.set_yticks(np.arange(pivot.shape[0]))
    ax.set_yticklabels([str(x).replace(', ', '\n') for x in pivot.index])
    ax.set_xticks(np.arange(pivot.shape[1]))
    ax.set_xticklabels([str(x).replace(', ', '\n') for x in pivot.columns], rotation=25, ha='right')
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = values[i, j]
            if pd.notna(value):
                ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=9)

fig, axes = plt.subplots(3, 2, figsize=(18, 16))

draw_heatmap(axes[0, 0], temp_wind, 'temp_label', 'wind_label', 'total_runs_mean', 'Total Runs: Temperature x Wind Speed')
draw_heatmap(axes[0, 1], temp_wind, 'temp_label', 'wind_label', 'strikeouts_mean', 'Strikeouts: Temperature x Wind Speed', cmap='YlGnBu')
draw_heatmap(axes[1, 0], temp_humidity, 'temp_label', 'humidity_label', 'total_runs_mean', 'Total Runs: Temperature x Humidity')
draw_heatmap(axes[1, 1], temp_humidity, 'temp_label', 'humidity_label', 'strikeouts_mean', 'Strikeouts: Temperature x Humidity', cmap='YlGnBu')
draw_heatmap(axes[2, 0], temp_pressure, 'temp_label', 'pressure_label', 'total_runs_mean', 'Total Runs: Temperature x Pressure')
draw_heatmap(axes[2, 1], temp_pressure, 'temp_label', 'pressure_label', 'strikeouts_mean', 'Strikeouts: Temperature x Pressure', cmap='YlGnBu')

for ax in axes[:, 0]:
    ax.set_ylabel('Temperature quintile')

plt.suptitle('Progressive Field: Does Temperature Work Differently Under Different Weather Contexts?', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pressure Bracket Validation: Recreate the Figure and Check the Math
pressure_summary = cle.groupby('pressure_bin', observed=True).agg(
    games=('rhum', 'size'),
    away_runs_mean=('away_runs_scored', 'mean'),
    away_runs_median=('away_runs_scored', 'median'),
    away_runs_std=('away_runs_scored', 'std'),
    share_6_plus=('away_runs_scored', lambda s: (s >= 6).mean()),
    total_runs_mean=('total_runs', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
    hits_mean=('hits', 'mean'),
    home_runs_mean=('home_runs_hit', 'mean'),
    walks_mean=('walks', 'mean'),
    temp_mean=('temp_f', 'mean'),
    humidity_mean=('rhum', 'mean'),
    pressure_mean=('pres', 'mean'),
    wind_mean=('wspd_mph', 'mean')
).reset_index()

overall_away_runs_mean = cle['away_runs_scored'].mean()
pressure_summary['away_runs_diff_from_mean'] = pressure_summary['away_runs_mean'] - overall_away_runs_mean
pressure_summary['pressure_range'] = pressure_summary['pressure_bin'].astype(str)
pressure_summary[['away_runs_mean', 'away_runs_diff_from_mean', 'away_runs_median', 'away_runs_std', 'share_6_plus', 'total_runs_mean', 'strikeouts_mean', 'hits_mean', 'home_runs_mean', 'walks_mean', 'temp_mean', 'humidity_mean', 'pressure_mean', 'wind_mean']] = pressure_summary[[
    'away_runs_mean', 'away_runs_diff_from_mean', 'away_runs_median', 'away_runs_std', 'share_6_plus', 'total_runs_mean', 'strikeouts_mean', 'hits_mean', 'home_runs_mean', 'walks_mean', 'temp_mean', 'humidity_mean', 'pressure_mean', 'wind_mean'
]].round(3)

print(f'Team-wide away-runs mean used by the heatmaps: {overall_away_runs_mean:.3f}')
print('The table below recreates the pressure-bracket means and the heatmap-style diff-from-mean values from the same Cleveland dataset used in the intro analysis.')
pressure_summary[['pressure_range', 'games', 'away_runs_mean', 'away_runs_diff_from_mean', 'away_runs_median', 'away_runs_std', 'share_6_plus', 'total_runs_mean', 'hits_mean', 'home_runs_mean', 'walks_mean', 'temp_mean', 'humidity_mean', 'pressure_mean', 'wind_mean', 'strikeouts_mean']]

In [ ]:
# What Seems to Drive the Central Pressure Spike?
pressure_order = cle['pressure_bin'].cat.categories
pressure_labels = [str(b).replace(', ', '\n').replace('(', '').replace(']', '') for b in pressure_order]
x = np.arange(len(pressure_order))

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0, 0].bar(x, pressure_summary['away_runs_diff_from_mean'], width=0.55, color='#1f77b4', label='Diff from overall mean')
axes[0, 0].plot(x, pressure_summary['away_runs_mean'], color='#ff7f0e', marker='o', linewidth=2.5, label='Raw away-runs mean')
axes[0, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0, 0].set_title('Away Runs by Pressure Bracket: Heatmap Metric vs Raw Mean')
axes[0, 0].set_ylabel('Runs')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(pressure_labels)
axes[0, 0].legend()

axes[0, 1].plot(x, pressure_summary['hits_mean'], marker='o', linewidth=2.5, color='#d62728', label='Hits')
axes[0, 1].plot(x, pressure_summary['home_runs_mean'], marker='^', linewidth=2.0, color='#9467bd', label='Home runs')
axes[0, 1].plot(x, pressure_summary['walks_mean'], marker='s', linewidth=2.0, color='#2ca02c', label='Walks')
axes[0, 1].set_title('Offensive Inputs by Pressure Bracket')
axes[0, 1].set_ylabel('Per-game average')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(pressure_labels)
axes[0, 1].legend()

axes[1, 0].plot(x, pressure_summary['temp_mean'], marker='o', linewidth=2.5, color='#ff7f0e', label='Temperature')
axes[1, 0].plot(x, pressure_summary['wind_mean'], marker='s', linewidth=2.5, color='#8c564b', label='Wind speed')
axes[1, 0].set_title('Temperature and Wind by Pressure Bracket')
axes[1, 0].set_ylabel('Degrees / mph')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(pressure_labels)
axes[1, 0].legend()

axes[1, 1].bar(x, pressure_summary['share_6_plus'], width=0.55, color='#17becf', label='Share of 6+ away-run games')
axes[1, 1].plot(x, pressure_summary['humidity_mean'], color='#7f7f7f', marker='o', linewidth=2.5, label='Humidity')
axes[1, 1].set_title('Humidity and Blowout Frequency by Pressure Bracket')
axes[1, 1].set_ylabel('Rate / hPa')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(pressure_labels)
axes[1, 1].legend()

for ax in axes.flat:
    ax.grid(alpha=0.25)

plt.suptitle('Progressive Field: The Central Pressure Bucket Can Be Checked Against Contact Quality and Other Weather Context', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pressure Crossover Diagnostics: Is the Central Bracket Riding Other Weather?
def pressure_crossover_summary(frame, other_bin, other_label):
    grouped = frame.groupby(['pressure_bin', other_bin], observed=True).agg(
        games=('game_pk', 'size'),
        away_runs_mean=('away_runs_scored', 'mean'),
        total_runs_mean=('total_runs', 'mean'),
        hits_mean=('hits', 'mean'),
        home_runs_mean=('home_runs_hit', 'mean')
    ).reset_index()
    grouped['pressure_label'] = grouped['pressure_bin'].astype(str)
    grouped[other_label] = grouped[other_bin].astype(str)
    return grouped

pressure_temp = pressure_crossover_summary(cle, 'temp_bin', 'temp_label')
pressure_humidity = pressure_crossover_summary(cle, 'humidity_bin', 'humidity_label')
pressure_wind = pressure_crossover_summary(cle, 'wind_bin', 'wind_label')

fig, axes = plt.subplots(3, 2, figsize=(18, 16))

draw_heatmap(axes[0, 0], pressure_temp, 'pressure_label', 'temp_label', 'away_runs_mean', 'Away Runs: Pressure x Temperature')
draw_heatmap(axes[0, 1], pressure_temp, 'pressure_label', 'temp_label', 'hits_mean', 'Hits: Pressure x Temperature', cmap='YlGnBu')
draw_heatmap(axes[1, 0], pressure_humidity, 'pressure_label', 'humidity_label', 'away_runs_mean', 'Away Runs: Pressure x Humidity')
draw_heatmap(axes[1, 1], pressure_humidity, 'pressure_label', 'humidity_label', 'home_runs_mean', 'Home Runs: Pressure x Humidity', cmap='YlGnBu')
draw_heatmap(axes[2, 0], pressure_wind, 'pressure_label', 'wind_label', 'away_runs_mean', 'Away Runs: Pressure x Wind Speed')
draw_heatmap(axes[2, 1], pressure_wind, 'pressure_label', 'wind_label', 'home_runs_mean', 'Home Runs: Pressure x Wind Speed', cmap='YlGnBu')

for ax in axes[:, 0]:
    ax.set_ylabel('Pressure bracket')

plt.suptitle('Progressive Field: The Central Pressure Bucket Can Be Compared Against Temperature, Humidity, and Wind Context', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

display(pressure_temp)
display(pressure_humidity)
display(pressure_wind)